In [17]:
!pip install streamlit pyngrok openai

In [18]:
!pip install -q google-generativeai

In [19]:
!streamlit run app.py &>/content/logs.txt &

In [38]:
from pyngrok import ngrok

ngrok.set_auth_token("3ES56KEZBTlew8AdynIKs0DISPK_3xExQBCbr2UVaxw5bsVnV")

In [41]:
public_url = ngrok.connect(8501)

print(public_url)

NgrokTunnel: "https://kabob-curdle-parka.ngrok-free.dev" -> "http://localhost:8501"


In [34]:
%%writefile app.py

Overwriting app.py


In [43]:
%%writefile app.py

import streamlit as st
import pandas as pd
import datetime
import plotly.express as px
import google.generativeai as genai
import json

genai.configure(api_key="AQ.Ab8RN6JoTSZD9sSyfE_rTGmoDCrnhaq8SwOKzMWY3Kd43N1t_A")

model = genai.GenerativeModel("gemini-2.5-flash")

# App Setup
st.set_page_config(
    page_title="Gut Health Tracker",
    layout="wide"
)

st.title("🌾 AI Gut Health Tracker")

# Navigation
page = st.sidebar.radio(
    "Navigation",
    ["Home", "Fiber Insights"]
)

# Session State
if "tracker_data" not in st.session_state:

    st.session_state.tracker_data = pd.DataFrame({
        "Date": [
            datetime.date(2026, 5, 25),
            datetime.date(2026, 5, 26),
            datetime.date(2026, 5, 27),
            datetime.date(2026, 5, 28),
            datetime.date(2026, 5, 29)
        ],
        "Water_Oz": [40, 80, 32, 96, 75],
        "Steps": [3000, 8000, 2500, 10000, 7000],
        "Pain_Scale": [8, 3, 9, 2, 4]
    })

if "fiber_log" not in st.session_state:

    st.session_state.fiber_log = pd.DataFrame(
        columns=[
            "Food",
            "Total_Fiber",
            "Soluble_Fiber",
            "Insoluble_Fiber"
        ]
    )

# Sidebar Form
st.sidebar.header("📥 Daily Log")

with st.sidebar.form("daily_form"):

    log_date = st.date_input(
        "Date",
        datetime.date.today()
    )

    water = st.number_input(
        "Water Intake (oz)",
        min_value=0,
        value=64
    )

    steps = st.number_input(
        "Steps",
        min_value=0,
        value=5000
    )

    pain = st.slider(
        "Pain Level",
        1,
        10,
        5
    )

    food = st.text_input(
        "🍎 Food Consumed",
        placeholder="Apple"
    )

    submit = st.form_submit_button(
        "Submit"
    )

# Handle Submission
if submit:

    new_entry = pd.DataFrame({
        "Date": [log_date],
        "Water_Oz": [water],
        "Steps": [steps],
        "Pain_Scale": [pain]
    })

    st.session_state.tracker_data = pd.concat(
        [st.session_state.tracker_data, new_entry],
        ignore_index=True
    )

    if food:

        prompt = f"""
Food item: {food}

Return ONLY valid JSON.

{{
  "food":"{food}",
  "total_fiber":0,
  "soluble_fiber":0,
  "insoluble_fiber":0
}}
"""

        try:

            response = model.generate_content(prompt)

            clean = response.text.replace(
                "```json",
                ""
            ).replace(
                "```",
                ""
            ).strip()

            fiber = json.loads(clean)

            fiber_entry = pd.DataFrame({
                "Food": [fiber["food"]],
                "Total_Fiber": [float(fiber["total_fiber"])],
                "Soluble_Fiber": [float(fiber["soluble_fiber"])],
                "Insoluble_Fiber": [float(fiber["insoluble_fiber"])]
            })

            st.session_state.fiber_log = pd.concat(
                [
                    st.session_state.fiber_log,
                    fiber_entry
                ],
                ignore_index=True
            )

        except Exception as e:

            st.error(
                f"Fiber Analysis Error: {e}"
            )

    st.success("Entry Logged!")

# Data
df = st.session_state.tracker_data
fiber_df = st.session_state.fiber_log

total_fiber = 0
total_soluble = 0
total_insoluble = 0

if not fiber_df.empty:

    total_fiber = fiber_df["Total_Fiber"].sum()
    total_soluble = fiber_df["Soluble_Fiber"].sum()
    total_insoluble = fiber_df["Insoluble_Fiber"].sum()

# HOME PAGE
if page == "Home":

    if not df.empty:

        col1, col2, col3 = st.columns(3)

        with col1:
            st.metric(
                "Pain Score",
                f"{df['Pain_Scale'].iloc[-1]}/10"
            )

        with col2:
            st.metric(
                "Water Intake",
                f"{df['Water_Oz'].iloc[-1]} oz"
            )

        with col3:
            st.metric(
                "Steps",
                f"{int(df['Steps'].iloc[-1]):,}"
            )

    st.write("---")

    st.subheader("🌾 Today's Fiber Summary")

    col1, col2 = st.columns(2)

    with col1:

        st.metric(
            "Total Fiber",
            f"{total_fiber:.1f} g"
        )

    goal = 25

    progress = min(
        total_fiber / goal,
        1.0
    )

    with col2:

        st.metric(
            "Goal Completion",
            f"{progress*100:.0f}%"
        )

    st.progress(progress)

    if total_fiber > 0:

        rec_prompt = f"""
Current Fiber Intake: {total_fiber:.1f} g

Goal: 25 g

Provide:
1. Food recommendation
2. Hydration recommendation
3. Lifestyle recommendation

Keep response under 50 words.
"""

        rec_response = model.generate_content(
            rec_prompt
        )

        st.subheader(
            "🤖 Personalized Recommendations"
        )

        st.write(
            rec_response.text
        )

    st.write("---")

    st.subheader("📈 Symptom Trends")

    fig = px.line(
        df,
        x="Date",
        y=["Water_Oz", "Pain_Scale"],
        markers=True
    )

    st.plotly_chart(
        fig,
        use_container_width=True
    )

# FIBER INSIGHTS PAGE
else:

    st.title("🌾 Fiber Insights")

    col1, col2, col3 = st.columns(3)

    with col1:
        st.metric(
            "Total Fiber",
            f"{total_fiber:.1f} g"
        )

    with col2:
        st.metric(
            "Soluble Fiber",
            f"{total_soluble:.1f} g"
        )

    with col3:
        st.metric(
            "Insoluble Fiber",
            f"{total_insoluble:.1f} g"
        )

    st.subheader(
        "Foods Logged"
    )

    st.dataframe(
        fiber_df,
        use_container_width=True
    )

    if not fiber_df.empty:

        fig2 = px.bar(
            fiber_df,
            x="Food",
            y="Total_Fiber",
            title="Fiber Contribution by Food"
        )

        st.plotly_chart(
            fig2,
            use_container_width=True
        )

Overwriting app.py


In [44]:
!streamlit run app.py &>/content/logs.txt &